# Structured Output

Not every agent action needs to be a tool call. Sometimes you need the LLM to produce a structured analysis, a classification result, or an execution plan — with type safety. The `output_schema` parameter makes a `CognitiveWorker` skip the tool-call loop entirely and output a typed Pydantic instance directly.

In this tutorial, we'll build a **requirements analysis agent** — it uses structured output to produce a typed analysis, Python code to process it, and then a regular think_unit to generate the final implementation plan. This demonstrates the powerful pattern of mixing LLM decisions with deterministic code.

## Initialize

First, let's set up the LLM and define the tools our planning agent will use.

In [ ]:
import os
from bridgic.model import OpenAILlm

api_key = os.environ.get("OPENAI_API_KEY", "your-api-key")
base_url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

llm = LLM(model="openai/gpt-4o-mini")

In [ ]:
from bridgic.core import tool


@tool(description="Search for technical documentation")
async def search_docs(query: str) -> str:
    return f"Documentation results for '{query}': Found API reference, tutorial, and examples"


@tool(description="Estimate implementation effort for a feature")
async def estimate_effort(feature: str, complexity: str) -> str:
    hours = {"low": "2-4", "medium": "8-16", "high": "24-40"}
    return f"Estimated effort for '{feature}' ({complexity}): {hours.get(complexity, '?')} hours"


@tool(description="Create a task in the project tracker")
async def create_task(title: str, priority: str, estimated_hours: str) -> str:
    return f"Created task: '{title}' (Priority: {priority}, Est: {estimated_hours}h)"

## Part 1: Structured Output with output_schema

A regular `CognitiveWorker` enters a tool-call loop: the LLM reasons, picks a tool, sees the result, and repeats until done. But when you set `output_schema`, the worker skips the loop entirely — it asks the LLM to fill a Pydantic model directly and returns a typed instance.

Let's define a Pydantic model for a requirements analysis result.

In [ ]:
from pydantic import BaseModel, Field
from typing import List


class Requirement(BaseModel):
    title: str = Field(description="Short title of the requirement")
    description: str = Field(description="Detailed description")
    priority: str = Field(description="Priority: high, medium, or low")
    complexity: str = Field(description="Complexity: high, medium, or low")
    dependencies: List[str] = Field(default_factory=list, description="Dependencies on other requirements")


class RequirementsAnalysis(BaseModel):
    project_name: str = Field(description="Name of the project")
    requirements: List[Requirement] = Field(description="List of analyzed requirements")
    summary: str = Field(description="Brief summary of the analysis")

Now create a `CognitiveWorker` with `output_schema`. This tells the framework: "Don't enter the tool-call loop — just ask the LLM to fill this model."

In [ ]:
from bridgic.amphibious import CognitiveWorker

# This worker produces a typed Pydantic instance instead of tool calls
analyzer_worker = CognitiveWorker.inline(
    "Analyze the project requirements. Break them down into individual requirements, "
    "assess priority and complexity for each, and identify dependencies.",
    output_schema=RequirementsAnalysis,
)

Use it in an agent — note that `max_attempts=1` makes sense here because structured output does not loop.

In [ ]:
from bridgic.amphibious import AmphibiousAutoma, CognitiveContext, think_unit


class RequirementsAnalyzer(AmphibiousAutoma[CognitiveContext]):
    analyzer = think_unit(analyzer_worker, max_attempts=1)

    async def on_agent(self, ctx: CognitiveContext):
        await self.analyzer

In [ ]:
agent = RequirementsAnalyzer(llm=llm)
result = await agent.arun(
    goal=(
        "Analyze requirements for a new e-commerce platform: "
        "user authentication, product catalog, shopping cart, "
        "payment processing, and order tracking"
    ),
)
print(result)

Unlike a regular think_unit (which enters a tool-call loop), a worker with `output_schema` asks the LLM to fill the Pydantic model directly. The result is a typed instance — `RequirementsAnalysis` in this case — that you can use in Python code with full type safety.

Key differences from the default behavior:

- **No tool-call loop**: the LLM produces the output in a single round.
- **Pydantic validation**: the output is validated against the schema automatically.
- **Type safety**: you get a real Python object with typed fields, not a free-form string.

## Part 2: Mixing LLM Decisions with Python Logic

The real power of structured output emerges when you combine it with code. The pattern is:

1. **LLM produces structured analysis** (output_schema) — the LLM reasons about the problem and fills a typed model.
2. **Python code processes the result** — deterministic filtering, sorting, enrichment.
3. **LLM acts on the processed result** (regular think_unit with tools) — the LLM takes actions based on the processed data.

This gives you the best of both worlds: LLM intelligence for complex reasoning and code reliability for data processing.

In [ ]:
class FullPlanningAgent(AmphibiousAutoma[CognitiveContext]):
    # Step 1: LLM produces structured analysis
    analyzer = think_unit(
        CognitiveWorker.inline(
            "Analyze the project requirements. Break them into individual items "
            "with priority, complexity, and dependencies.",
            output_schema=RequirementsAnalysis,
        ),
        max_attempts=1,
    )

    # Step 3: LLM creates tasks based on processed requirements
    task_creator = think_unit(
        CognitiveWorker.inline(
            "Create project tasks for each high-priority requirement. "
            "Estimate effort and create tasks in the tracker."
        ),
        max_attempts=10,
    )

    async def on_agent(self, ctx: CognitiveContext):
        # Step 1: Get structured analysis from LLM
        await self.analyzer

        # Step 2: Python code processes the result
        # (In practice, you'd access the structured output from context/history)
        # Here we show the pattern of interleaving code between think units

        # Step 3: LLM creates tasks using tools
        await self.task_creator

In [ ]:
agent = FullPlanningAgent(llm=llm)
result = await agent.arun(
    goal=(
        "Analyze and plan the e-commerce platform project: "
        "authentication, catalog, cart, payments, order tracking. "
        "Focus on creating tasks for high-priority items first."
    ),
    tools=[search_docs, estimate_effort, create_task],
)
print(result)

This agent works in two distinct phases:

1. **Structured analysis** (`self.analyzer`): The LLM fills the `RequirementsAnalysis` model — a single LLM call, no tools involved. The result is a typed object with project name, individual requirements (each with priority, complexity, and dependencies), and a summary.

2. **Task creation** (`self.task_creator`): A regular think_unit that enters the tool-call loop. The LLM sees the analysis from step 1 in its context and uses the `estimate_effort` and `create_task` tools to turn the analysis into actionable tasks.

Between these two steps, you can insert any Python code — filtering out low-priority items, sorting by dependencies, enriching with data from external systems. The structured output gives you a typed object to work with, not a free-form string that needs parsing.

## When to Use output_schema vs Tool Calls

| Aspect | `output_schema` | Tool Calls (default) |
|--------|-----------------|---------------------|
| **Output** | Typed Pydantic instance | Tool execution results |
| **LLM Rounds** | Single round | Multiple rounds (loop) |
| **Use Case** | Analysis, planning, classification | Taking actions (API calls, file ops) |
| **Type Safety** | Full (Pydantic validation) | Depends on tool return types |
| **Cost** | Lower (one LLM call) | Higher (multiple calls) |

Use `output_schema` for tasks where you want a structured *decision* rather than an *action*: planning, analysis, classification, extraction, evaluation. Use regular tool calls when the agent needs to *do* something — call APIs, read files, interact with external systems.

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we explored structured output and hybrid orchestration:

- **`output_schema`** on a CognitiveWorker makes it produce a typed Pydantic instance instead of entering the tool-call loop. Set it with `CognitiveWorker.inline("...", output_schema=MyModel)`.
- Structured output is ideal for **analysis, planning, classification, and extraction** — tasks where you need a structured decision rather than an action.
- The **LLM + Code hybrid pattern** is powerful: use structured output for LLM decisions, Python code for deterministic processing, and regular think_units for tool-based actions.
- This gives you the best of both worlds: **LLM intelligence** for complex reasoning and **code reliability** for data processing.

Structured output turns the LLM from a tool-calling engine into a structured reasoning engine, opening up new patterns for agent design.